# Clausius Asymmetric Quarkyonic Workflow

This notebook implements the Clausius case requested for asymmetric quarkyonic nuclear matter in `QMM`.

It scans the Clausius parameter `c` through a target incompressibility `K_0`, then computes:

- symmetric ground-state parameters `a`, `b`, `c`
- asymmetric parameters `a_n`, `a_{pn}`, `b_n`, `b_{pn}`
- consistency with the symmetric limits
- fixed-`y` asymmetric quarkyonic profiles for `y = 0.0, 0.1, ..., 0.5`
- beta-equilibrium quarkyonic EOS with leptons
- neutron-star mass-radius sequences

This notebook uses the pure `clausius` model in `QMM`, meaning:

- Clausius attractive mean field
- standard van der Waals excluded volume for the repulsive sector

If instead you want Clausius attraction with Carnahan-Starling or TVM repulsion, use the separate hybrid notebooks already added in `QMM`.


## Physics Setup

The asymmetric attractive contribution is handled in the Clausius form

$$
\varepsilon_{\mathrm{attr}}(n_p,n_n) =
-\frac{a_n(n_p^2+n_n^2)+2a_{pn}n_p n_n}{1+c n_B},
\qquad n_B=n_p+n_n.
$$

In the symmetric limit `y = 1/2`, the notebook checks that

$$
a = \frac{a_n+a_{pn}}{2}, \qquad b = \frac{b_n+b_{pn}}{2}.
$$

The constraints imposed here are:

- `E/A = -16 MeV`
- `n_0 = 0.16 fm^{-3}`
- `J = 32.5 MeV`
- `L = 58.9 MeV`
- target `K_0` values in the Clausius range


In [1]:
from copy import deepcopy
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython import get_ipython
from IPython.display import display

ip = get_ipython()
if ip is not None:
    try:
        ip.run_line_magic('matplotlib', 'inline')
    except Exception:
        pass


def locate_root() -> Path:
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / 'src' / 'qmm').exists():
            return candidate.resolve()
    raise RuntimeError('Could not locate the QMM project root from this notebook.')


ROOT = locate_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from qmm.config import load_run_config
from qmm.runner import run_workflow

BASE_CONFIG_PATH = ROOT / 'examples' / 'clausius_asymmetric_base.json'
OUTPUT_ROOT = ROOT / 'results' / 'generated' / 'clausius_asymmetric_scan_lambda200_smooth'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

with BASE_CONFIG_PATH.open('r', encoding='utf-8') as handle:
    base_config = json.load(handle)

print('ROOT =', ROOT)
print('Base config =', BASE_CONFIG_PATH)
print(json.dumps(base_config, indent=2))


ROOT = /Users/dajuarez4/Documents/QuarkMatt/good_repo/QMM
Base config = /Users/dajuarez4/Documents/QuarkMatt/good_repo/QMM/examples/clausius_asymmetric_base.json
{
  "run_name": "clausius_asymmetric",
  "model": {
    "name": "clausius",
    "parameter_search": {
      "enabled": true,
      "target_k0": 280.0,
      "parameter_min": 0.0,
      "parameter_max": 4.74,
      "scan_steps": 121,
      "tol": 1e-08
    }
  },
  "workflows": {
    "ground_state": true,
    "quantum_critical": false,
    "hadronic_eos_table": true,
    "symmetric_quarkyonic": false,
    "asymmetric_fit": true,
    "asymmetric_quarkyonic": true,
    "neutron_star": true
  },
  "physical": {
    "n0": 0.16,
    "binding_energy": -16.0,
    "symmetry_energy_j": 32.5,
    "symmetry_slope_l": 58.9
  },
  "hadronic_eos": {
    "n_min_ratio": 0.2,
    "n_max_ratio": 5.0,
    "n_points": 160,
    "smoothing_window": 9,
    "smoothing_degree": 3,
    "derivative_floor": 1e-10
  },
  "quarkyonic": {
    "n_min_ratio": 

## Scan Inputs

The default targets span the requested incompressibility window while keeping the runtime moderate.
Edit the next cell if you want a denser scan.


In [2]:
# Example smaller subset:
# target_k0_values = [250.0, 280.0, 315.0]
target_k0_values = [250.0, 260.0, 270.0, 280.0, 290.0, 300.0, 315.0]

proton_fraction_values = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
lambda_momentum_mev = 200.0
branch_mode = 'target_l'
write_plots = True

print('K0 targets:', target_k0_values)
print('fixed-y values:', proton_fraction_values)
print('lambda [MeV]:', lambda_momentum_mev)
print('branch_mode:', branch_mode)


K0 targets: [250.0, 260.0, 270.0, 280.0, 290.0, 300.0, 315.0]
fixed-y values: [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
lambda [MeV]: 200.0
branch_mode: target_l


## Build One Config Per Target

The helper below starts from the executable Clausius base JSON and only overrides the run-specific pieces.


In [3]:
def build_config_for_target_k0(target_k0: float) -> dict:
    config = deepcopy(base_config)
    run_slug = f'clausius_k0_{int(round(target_k0))}'

    config['run_name'] = run_slug
    config['model']['name'] = 'clausius'
    config['model']['parameter_search']['enabled'] = True
    config['model']['parameter_search']['target_k0'] = float(target_k0)
    config['model']['parameter_search']['parameter_min'] = 0.0
    config['model']['parameter_search']['parameter_max'] = 4.74
    config['model']['parameter_search']['scan_steps'] = 121

    config['workflows']['ground_state'] = True
    config['workflows']['quantum_critical'] = False
    config['workflows']['hadronic_eos_table'] = True
    config['workflows']['symmetric_quarkyonic'] = False
    config['workflows']['asymmetric_fit'] = True
    config['workflows']['asymmetric_quarkyonic'] = True
    config['workflows']['neutron_star'] = True

    config['quarkyonic']['lambda_momentum_mev'] = float(lambda_momentum_mev)
    config['quarkyonic']['n_min_ratio'] = 1.0e-6
    config['quarkyonic']['n_max_ratio'] = 5.0
    config['quarkyonic']['n_points'] = 60
    config['quarkyonic']['fq_scan_points'] = 61
    config['quarkyonic']['shell_integral_points'] = 100
    config['quarkyonic']['quark_integral_points'] = 100
    config['asymmetric']['enabled'] = True
    config['asymmetric']['branch_mode'] = branch_mode
    config['asymmetric']['target_j'] = 32.5
    config['asymmetric']['target_l'] = 58.9
    config['asymmetric']['target_k0'] = float(target_k0)
    config['asymmetric']['proton_fraction_values'] = list(proton_fraction_values)
    config['asymmetric']['beta_equilibrium'] = True

    run_dir = OUTPUT_ROOT / f'K0_{int(round(target_k0))}'
    config['output']['directory'] = str(run_dir)
    config['output']['write_json'] = True
    config['output']['write_csv'] = True
    config['output']['write_plots'] = bool(write_plots)
    config['neutron_star']['sequence_points'] = 12

    return config


## Run The Scan

This writes one JSON config per target `K0`, runs the Clausius workflow, and stores the returned summaries.


In [4]:
scan_summaries = {}
failures = []
config_dir = OUTPUT_ROOT / 'configs'
config_dir.mkdir(parents=True, exist_ok=True)

for target_k0 in target_k0_values:
    config_dict = build_config_for_target_k0(target_k0)
    config_path = config_dir / f'clausius_k0_{int(round(target_k0))}.json'
    config_path.write_text(json.dumps(config_dict, indent=2), encoding='utf-8')

    try:
        run_config = load_run_config(config_path)
        summary = run_workflow(run_config)
        scan_summaries[target_k0] = summary
        print(f'finished K0={target_k0:.1f} MeV -> c={summary["ground_state"]["parameter_value"]:.6f}')
    except Exception as exc:
        failures.append((target_k0, str(exc)))
        print(f'failed K0={target_k0:.1f} MeV -> {exc}')

if failures:
    print('failures detected:')
    for item in failures:
        print(item)
else:
    print('all runs completed')


finished K0=250.0 MeV -> c=4.735853


finished K0=260.0 MeV -> c=4.516312


finished K0=270.0 MeV -> c=4.309453


finished K0=280.0 MeV -> c=4.114108


finished K0=290.0 MeV -> c=3.929255


finished K0=300.0 MeV -> c=3.753994


finished K0=315.0 MeV -> c=3.507371
all runs completed


## Collect Summaries

The next cell extracts the main outputs you asked for:

- `a`, `b`, `c`, `K0`
- `a_n`, `a_pn`, `b_n`, `b_pn`
- `a_pn/a_n`, `b_pn/b_n`
- neutron-star maximum mass and radius at maximum mass


In [5]:
ground_rows = []
asym_rows = []
ns_rows = []

for target_k0, summary in sorted(scan_summaries.items()):
    ground = dict(summary['ground_state'])
    ground['target_k0_requested'] = target_k0
    ground_rows.append(ground)

    asym = dict(summary['asymmetric_fit'])
    asym['target_k0_requested'] = target_k0
    asym_rows.append(asym)

    ns = dict(summary['neutron_star'])
    ns['target_k0_requested'] = target_k0
    ns_rows.append(ns)

ground_df = pd.DataFrame(ground_rows).sort_values('target_k0_requested').reset_index(drop=True)
asym_df = pd.DataFrame(asym_rows).sort_values('target_k0_requested').reset_index(drop=True)
ns_df = pd.DataFrame(ns_rows).sort_values('target_k0_requested').reset_index(drop=True)

consistency_df = asym_df[['target_k0_requested', 'a_avg', 'b_avg', 'a_n', 'a_pn', 'b_n', 'b_pn']].copy()
consistency_df['a_from_split'] = 0.5 * (consistency_df['a_n'] + consistency_df['a_pn'])
consistency_df['b_from_split'] = 0.5 * (consistency_df['b_n'] + consistency_df['b_pn'])
consistency_df['a_avg_minus_split'] = consistency_df['a_avg'] - consistency_df['a_from_split']
consistency_df['b_avg_minus_split'] = consistency_df['b_avg'] - consistency_df['b_from_split']

ground_df.to_csv(OUTPUT_ROOT / 'ground_state_summary.csv', index=False)
asym_df.to_csv(OUTPUT_ROOT / 'asymmetric_fit_summary.csv', index=False)
ns_df.to_csv(OUTPUT_ROOT / 'neutron_star_summary.csv', index=False)
consistency_df.to_csv(OUTPUT_ROOT / 'symmetric_limit_consistency.csv', index=False)

display(ground_df[['target_k0_requested', 'parameter_value', 'a', 'b', 'K0', 'n_sat', 'binding']])
display(asym_df[['target_k0_requested', 'a_n', 'a_pn', 'b_n', 'b_pn', 'ratio_a_pn_over_a_n', 'ratio_b_pn_over_b_n', 'J', 'L']])
display(ns_df[['target_k0_requested', 'max_mass_msun', 'radius_at_max_mass_km', 'max_mass_at_upper_eos_boundary']])
display(consistency_df[['target_k0_requested', 'a_avg', 'a_from_split', 'a_avg_minus_split', 'b_avg', 'b_from_split', 'b_avg_minus_split']])


,target_k0_requested,parameter_value,a,b,K0,n_sat,binding
0,250.0,4.735853,472.327509,1.727371,250.000001,0.16,-16.0
1,260.0,4.516312,466.026020,1.800838,260.000002,0.16,-16.0
2,270.0,4.309453,460.064471,1.870418,269.999998,0.16,-16.0
3,280.0,4.114108,454.412697,1.936450,280.000001,0.16,-16.0
4,290.0,3.929255,449.044179,1.999231,290.000000,0.16,-16.0
5,300.0,3.753994,443.935490,2.059026,300.000002,0.16,-16.0
6,315.0,3.507371,436.714809,2.143624,314.999996,0.16,-16.0


,target_k0_requested,a_n,a_pn,b_n,b_pn,ratio_a_pn_over_a_n,ratio_b_pn_over_b_n,J,L
0,250.0,259.413733,685.241284,1.316923,2.137819,2.641500,1.623344,32.5,58.9
1,260.0,256.197400,675.854641,1.351272,2.250404,2.638023,1.665397,32.5,58.9
2,270.0,253.177801,666.951141,1.384757,2.356079,2.634319,1.701439,32.5,58.9
3,280.0,250.336526,658.488869,1.417423,2.455476,2.630415,1.732352,32.5,58.9
4,290.0,247.657455,650.430903,1.449313,2.549149,2.626333,1.758867,32.5,58.9
5,300.0,245.126408,642.744572,1.480464,2.637588,2.622094,1.781596,32.5,58.9
6,315.0,241.580371,631.849247,1.525881,2.761368,2.615483,1.809688,32.5,58.9


,target_k0_requested,max_mass_msun,radius_at_max_mass_km,max_mass_at_upper_eos_boundary
0,250.0,2.678207,12.279080,True
1,260.0,2.722179,12.489312,True
2,270.0,2.759856,12.676950,True
3,280.0,2.792188,12.847532,True
4,290.0,2.820064,13.001345,True
5,300.0,2.844156,13.142153,True
6,315.0,2.874580,13.327683,True


,target_k0_requested,a_avg,a_from_split,a_avg_minus_split,b_avg,b_from_split,b_avg_minus_split
0,250.0,472.327509,472.327509,0.000000e+00,1.727371,1.727371,0.0
1,260.0,466.026020,466.026020,0.000000e+00,1.800838,1.800838,0.0
2,270.0,460.064471,460.064471,5.684342e-14,1.870418,1.870418,0.0
3,280.0,454.412697,454.412697,0.000000e+00,1.936450,1.936450,0.0
4,290.0,449.044179,449.044179,0.000000e+00,1.999231,1.999231,0.0
5,300.0,443.935490,443.935490,5.684342e-14,2.059026,2.059026,0.0
6,315.0,436.714809,436.714809,0.000000e+00,2.143624,2.143624,0.0


## Parameter Plots


In [6]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)

axes[0, 0].plot(ground_df['target_k0_requested'], ground_df['parameter_value'], marker='o', lw=2)
axes[0, 0].set_xlabel(r'target $K_0$ [MeV]')
axes[0, 0].set_ylabel('resolved c')
axes[0, 0].set_title('Clausius parameter from target K0')

axes[0, 1].plot(ground_df['target_k0_requested'], ground_df['a'], marker='o', lw=2, label='a')
axes[0, 1].plot(ground_df['target_k0_requested'], ground_df['b'], marker='s', lw=2, label='b')
axes[0, 1].set_xlabel(r'target $K_0$ [MeV]')
axes[0, 1].set_title('Symmetric nuclear-matter couplings')
axes[0, 1].legend(loc='best')

axes[1, 0].plot(asym_df['target_k0_requested'], asym_df['a_n'], marker='o', lw=2, label=r'$a_n$')
axes[1, 0].plot(asym_df['target_k0_requested'], asym_df['a_pn'], marker='s', lw=2, label=r'$a_{pn}$')
axes[1, 0].set_xlabel(r'target $K_0$ [MeV]')
axes[1, 0].set_ylabel(r'[MeV fm$^3$]')
axes[1, 0].set_title('Attractive asymmetric couplings')
axes[1, 0].legend(loc='best')

axes[1, 1].plot(asym_df['target_k0_requested'], asym_df['b_n'], marker='o', lw=2, label=r'$b_n$')
axes[1, 1].plot(asym_df['target_k0_requested'], asym_df['b_pn'], marker='s', lw=2, label=r'$b_{pn}$')
axes[1, 1].set_xlabel(r'target $K_0$ [MeV]')
axes[1, 1].set_ylabel(r'[fm$^3$]')
axes[1, 1].set_title('Repulsive asymmetric couplings')
axes[1, 1].legend(loc='best')

plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), constrained_layout=True)
axes[0].plot(asym_df['target_k0_requested'], asym_df['ratio_a_pn_over_a_n'], marker='o', lw=2)
axes[0].set_xlabel(r'target $K_0$ [MeV]')
axes[0].set_ylabel(r'$a_{pn}/a_n$')
axes[0].set_title('Attractive split ratio')

axes[1].plot(asym_df['target_k0_requested'], asym_df['ratio_b_pn_over_b_n'], marker='o', lw=2)
axes[1].set_xlabel(r'target $K_0$ [MeV]')
axes[1].set_ylabel(r'$b_{pn}/b_n$')
axes[1].set_title('Repulsive split ratio')

plt.show()


/var/folders/vw/wh6xbs2n4y570pctw6_y3t0r0000gp/T/ipykernel_39834/3973543346.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/vw/wh6xbs2n4y570pctw6_y3t0r0000gp/T/ipykernel_39834/3973543346.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Neutron-Star Summary

This summarizes the beta-equilibrium quarkyonic EOS at the neutron-star level.


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)

axes[0].plot(ns_df['target_k0_requested'], ns_df['max_mass_msun'], marker='o', lw=2)
axes[0].axhline(2.0, color='gray', ls='--', lw=1.0)
axes[0].set_xlabel(r'target $K_0$ [MeV]')
axes[0].set_ylabel(r'$M_{\max}$ [$M_\odot$]')
axes[0].set_title('Maximum neutron-star mass')

axes[1].plot(ns_df['target_k0_requested'], ns_df['radius_at_max_mass_km'], marker='o', lw=2)
axes[1].set_xlabel(r'target $K_0$ [MeV]')
axes[1].set_ylabel(r'$R(M_{\max})$ [km]')
axes[1].set_title('Radius at maximum mass')

plt.show()


/var/folders/vw/wh6xbs2n4y570pctw6_y3t0r0000gp/T/ipykernel_39834/271539484.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Pure Hadronic EOS For One Target

The next cells inspect the pure Clausius hadronic EOS before quarkyonic matter is introduced.


In [8]:
selected_target_k0_hadronic = target_k0_values[0]
hadronic_summary = scan_summaries[selected_target_k0_hadronic]
hadronic_csv = Path(hadronic_summary['outputs']['hadronic_eos_csv'])
hadronic_df = pd.read_csv(hadronic_csv)

print('Hadronic EOS CSV:', hadronic_csv)
display(hadronic_df.head())


Hadronic EOS CSV: /Users/dajuarez4/Documents/QuarkMatt/good_repo/QMM/results/generated/clausius_asymmetric_scan_lambda200_smooth/K0_250/clausius_k0_250_hadronic_eos.csv


,model,parameter_name,parameter_value,a,b,K0,n,n_over_n0,eps_raw,eps,e_per_particle_minus_m,mu_b,pressure,dP_dn,d2eps_dn2,vs2
0,clausius,c,4.735853,472.327509,1.727371,250.000001,0.032000,0.200000,29.846282,29.846282,-5.303683,926.788225,-0.189059,-10.100166,-315.630178,-0.010898
1,clausius,c,4.735853,472.327509,1.727371,250.000001,0.036830,0.230189,34.319268,34.319263,-6.175604,925.328992,-0.239222,-10.618912,-288.320856,-0.011476
2,clausius,c,4.735853,472.327509,1.727371,250.000001,0.041660,0.260377,38.785513,38.785514,-7.007162,924.001893,-0.291247,-10.906497,-261.795454,-0.011804
3,clausius,c,4.735853,472.327509,1.727371,250.000001,0.046491,0.290566,43.245654,43.245670,-7.797209,922.802413,-0.344063,-10.975636,-236.083073,-0.011894
4,clausius,c,4.735853,472.327509,1.727371,250.000001,0.051321,0.320755,47.700308,47.700340,-8.545463,921.725864,-0.396673,-10.838474,-211.190845,-0.011759


In [9]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), constrained_layout=True)

axes[0].plot(hadronic_df['n_over_n0'], hadronic_df['e_per_particle_minus_m'], lw=2.2)
axes[0].axhline(0.0, color='black', ls='--', lw=1.0)
axes[0].set_xlabel(r'$n/n_0$')
axes[0].set_ylabel(r'$E/A - m_N$ [MeV]')
axes[0].set_title(f'Hadronic binding curve, K0={selected_target_k0_hadronic:.0f} MeV')

axes[1].plot(hadronic_df['n_over_n0'], hadronic_df['pressure'], lw=2.2)
axes[1].axhline(0.0, color='black', ls='--', lw=1.0)
axes[1].set_xlabel(r'$n/n_0$')
axes[1].set_ylabel(r'$P$ [MeV fm$^{-3}$]')
axes[1].set_title('Pure hadronic pressure')

axes[2].plot(hadronic_df['n_over_n0'], hadronic_df['vs2'], lw=2.2)
axes[2].axhline(1.0 / 3.0, color='gray', ls=(0, (5, 5)), lw=1.0)
axes[2].set_ylim(-0.1, 0.8)
axes[2].set_xlabel(r'$n/n_0$')
axes[2].set_ylabel(r'$v_s^2$')
axes[2].set_title('Pure hadronic sound speed')

plt.show()


/var/folders/vw/wh6xbs2n4y570pctw6_y3t0r0000gp/T/ipykernel_39834/1287798810.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Beta-Equilibrium Quarkyonic EOS And Mass-Radius Curve

Pick one target `K0` below and the notebook will load the beta-equilibrium EOS and neutron-star sequence from that run.


In [10]:
selected_target_k0_beta = target_k0_values[0]
selected_summary = scan_summaries[selected_target_k0_beta]

beta_csv = Path(selected_summary['outputs']['asymmetric_beta_csv'])
ns_csv = Path(selected_summary['outputs']['neutron_star_sequence_csv'])

beta_df = pd.read_csv(beta_csv)
ns_sequence_df = pd.read_csv(ns_csv)

print('Beta-equilibrium CSV:', beta_csv)
print('Neutron-star sequence CSV:', ns_csv)
display(beta_df.head())
display(ns_sequence_df.head())


Beta-equilibrium CSV: /Users/dajuarez4/Documents/QuarkMatt/good_repo/QMM/results/generated/clausius_asymmetric_scan_lambda200_smooth/K0_250/clausius_k0_250_asymmetric_beta.csv
Neutron-star sequence CSV: /Users/dajuarez4/Documents/QuarkMatt/good_repo/QMM/results/generated/clausius_asymmetric_scan_lambda200_smooth/K0_250/clausius_k0_250_neutron_star_sequence.csv


,model,momentum_mode,parameter_name,parameter_value,n_b,n_over_n0,y,quark_fraction,k_bu,k_fp,k_fn,n_p,n_n,energy_density,mu_e,eps_e,eps_mu,mu_b,pressure,vs2
0,clausius,quarkyonic,c,4.735853,1.600000e-07,0.000001,0.000001,0.994176,0.004311,0.004311,0.004758,9.318625e-16,9.318615e-10,0.000150,0.105885,0.000000,0.0,940.405545,0.000187,3.096907e-08
1,clausius,quarkyonic,c,4.735853,1.355948e-02,0.084747,0.003272,0.003143,0.070505,0.119324,0.740778,4.422888e-05,1.347263e-02,12.766260,21.662694,0.000726,0.0,942.619373,0.013289,1.931190e-03
2,clausius,quarkyonic,c,4.735853,2.711880e-02,0.169492,0.008347,0.000835,0.051334,0.193475,0.938011,2.261688e-04,2.686998e-02,25.562940,37.228408,0.006331,0.0,944.246150,0.045107,3.433631e-03
3,clausius,quarkyonic,c,4.735853,4.067812e-02,0.254238,0.013748,0.000349,0.040611,0.262979,1.078743,5.590642e-04,4.010484e-02,38.377966,50.310058,0.021118,0.0,945.768171,0.096593,5.243040e-03
4,clausius,quarkyonic,c,4.735853,5.423744e-02,0.338984,0.019013,0.000183,0.033857,0.325655,1.192969,1.031016e-03,5.319652e-02,51.212175,61.684523,0.047726,0.0,947.406898,0.175148,7.640794e-03


,model,parameter_name,parameter_value,central_pressure_mev_fm3,central_energy_density_mev_fm3,mass_msun,radius_km
0,clausius,c,4.735853,0.100000,39.028220,0.007127,3.917050
1,clausius,c,4.735853,0.210806,55.939272,0.022486,5.402162
2,clausius,c,4.735853,0.444392,77.396462,0.046766,6.440673
3,clausius,c,4.735853,0.936805,102.860664,0.087811,7.371425
4,clausius,c,4.735853,1.974840,135.626973,0.157094,8.268668


In [11]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)

axes[0, 0].plot(beta_df['n_over_n0'], beta_df['y'], lw=2)
axes[0, 0].set_xlabel(r'$n/n_0$')
axes[0, 0].set_ylabel(r'$y$')
axes[0, 0].set_title('Beta-equilibrium proton fraction')

axes[0, 1].plot(beta_df['n_over_n0'], beta_df['quark_fraction'], lw=2)
axes[0, 1].set_xlabel(r'$n/n_0$')
axes[0, 1].set_ylabel(r'$f_Q$')
axes[0, 1].set_title('Beta-equilibrium quark fraction')

beta_energy_per_baryon_gev = (beta_df['energy_density'] / beta_df['n_b'] - 938.0) / 1000.0
axes[1, 0].plot(beta_df['n_over_n0'], beta_energy_per_baryon_gev, lw=2)
axes[1, 0].set_xlabel(r'$n/n_0$')
axes[1, 0].set_ylabel(r'$\epsilon/\rho_B - m_N$ [GeV]')
axes[1, 0].set_title('Beta-equilibrium energy per baryon')

axes[1, 1].plot(ns_sequence_df['radius_km'], ns_sequence_df['mass_msun'], lw=2)
axes[1, 1].set_xlabel(r'$R$ [km]')
axes[1, 1].set_ylabel(r'$M$ [$M_\odot$]')
axes[1, 1].set_title(f'Mass-radius curve, K0={selected_target_k0_beta:.0f} MeV')

plt.show()

fig, ax = plt.subplots(figsize=(6.5, 4.6), constrained_layout=True)
ax.plot(beta_df['n_over_n0'], beta_df['vs2'], lw=2)
ax.axhline(1.0 / 3.0, color='gray', ls=(0, (5, 5)), lw=1.0)
selected_vs2_upper = max(1.0, 1.05 * beta_df['vs2'].max())
ax.set_ylim(-0.1, selected_vs2_upper)
ax.set_xlabel(r'$n/n_0$')
ax.set_ylabel(r'$v_s^2$')
ax.set_title(f'Beta-equilibrium sound speed, K0={selected_target_k0_beta:.0f} MeV')
plt.show()


/var/folders/vw/wh6xbs2n4y570pctw6_y3t0r0000gp/T/ipykernel_39834/2279051969.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/vw/wh6xbs2n4y570pctw6_y3t0r0000gp/T/ipykernel_39834/2279051969.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Beta-Equilibrium Overlay For Both Asymmetric Branches

This section overlays the beta-equilibrium Clausius results for all seven incompressibility targets in both asymmetric branches.
The color identifies `K_0`, the solid lines are for `b_{pn} \\neq b_n`, and the dashed lines are for `b_{pn}=b_n`.


In [12]:
comparison_target_k0_values = [250.0, 260.0, 270.0, 280.0, 290.0, 300.0, 315.0]
beta_comparison_unequal = {}
beta_comparison_equal = {}

def blend_with_white(color, blend=0.38):
    red, green, blue, alpha = color
    return (
        red + (1.0 - red) * blend,
        green + (1.0 - green) * blend,
        blue + (1.0 - blue) * blend,
        alpha,
    )

for target_k0 in comparison_target_k0_values:
    summary = scan_summaries.get(target_k0)
    if summary is None:
        summary_path = OUTPUT_ROOT / f'K0_{int(round(target_k0))}' / f'clausius_k0_{int(round(target_k0))}_summary.json'
        if not summary_path.exists():
            raise FileNotFoundError(f'Missing summary for K0={target_k0:.1f} MeV: {summary_path}')
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
    beta_csv = Path(summary['outputs']['asymmetric_beta_csv'])
    beta_comparison_unequal[target_k0] = pd.read_csv(beta_csv)

    equal_beta_csv = ROOT / 'results' / 'generated' / 'clausius_equal_b_scan_fullrange_lambda200_smooth' / f'K0_{int(round(target_k0))}' / f'clausius_equal_b_k0_{int(round(target_k0))}_asymmetric_beta.csv'
    if not equal_beta_csv.exists():
        raise FileNotFoundError(f'Missing equal-b beta CSV for K0={target_k0:.1f} MeV: {equal_beta_csv}')
    beta_comparison_equal[target_k0] = pd.read_csv(equal_beta_csv)

fig, axes = plt.subplots(2, 2, figsize=(13, 10), constrained_layout=True)
base_colors = plt.cm.viridis([index / max(len(comparison_target_k0_values) - 1, 1) for index in range(len(comparison_target_k0_values))])
equal_colors = [blend_with_white(color) for color in base_colors]

for base_color, equal_color, target_k0 in zip(base_colors, equal_colors, comparison_target_k0_values):
    df_unequal = beta_comparison_unequal[target_k0]
    df_equal = beta_comparison_equal[target_k0]
    label = f'K0={target_k0:.0f} MeV'
    energy_per_baryon_unequal_gev = (df_unequal['energy_density'] / df_unequal['n_b'] - 938.0) / 1000.0
    energy_per_baryon_equal_gev = (df_equal['energy_density'] / df_equal['n_b'] - 938.0) / 1000.0
    axes[0, 0].plot(df_unequal['n_over_n0'], df_unequal['quark_fraction'], lw=2.2, color=base_color, ls='-', label=label)
    axes[0, 0].plot(df_equal['n_over_n0'], df_equal['quark_fraction'], lw=2.2, color=equal_color, ls='--')
    axes[0, 1].plot(df_unequal['n_over_n0'], df_unequal['vs2'], lw=2.2, color=base_color, ls='-')
    axes[0, 1].plot(df_equal['n_over_n0'], df_equal['vs2'], lw=2.2, color=equal_color, ls='--')
    axes[1, 0].plot(df_unequal['n_over_n0'], energy_per_baryon_unequal_gev, lw=2.2, color=base_color, ls='-')
    axes[1, 0].plot(df_equal['n_over_n0'], energy_per_baryon_equal_gev, lw=2.2, color=equal_color, ls='--')
    axes[1, 1].plot(df_unequal['n_over_n0'], df_unequal['pressure'], lw=2.2, color=base_color, ls='-')
    axes[1, 1].plot(df_equal['n_over_n0'], df_equal['pressure'], lw=2.2, color=equal_color, ls='--')

axes[0, 0].set_xlabel(r'$n/n_0$')
axes[0, 0].set_ylabel(r'$f_Q$')
axes[0, 0].set_title('Beta-equilibrium quark fraction')
k0_legend = axes[0, 0].legend(loc='upper left', fontsize=9, title=r'$K_0$')
style_handles = [
    axes[0, 0].plot([], [], color='black', lw=2.2, ls='-', label=r'$b_{pn} \neq b_n$')[0],
    axes[0, 0].plot([], [], color='black', lw=2.2, ls='--', label=r'$b_{pn} = b_n$')[0],
]
style_legend = axes[0, 0].legend(handles=style_handles, loc='lower right', fontsize=9, title='Branch')
axes[0, 0].add_artist(k0_legend)

axes[0, 1].set_xlabel(r'$n/n_0$')
axes[0, 1].set_ylabel(r'$v_s^2$')
axes[0, 1].axhline(1.0 / 3.0, color='gray', ls=(0, (5, 5)), lw=1.0)
overlay_vs2_upper = max(
    1.0,
    1.05 * max(df['vs2'].max() for df in beta_comparison_unequal.values()),
    1.05 * max(df['vs2'].max() for df in beta_comparison_equal.values()),
)
axes[0, 1].set_ylim(-0.1, overlay_vs2_upper)
axes[0, 1].set_title('Beta-equilibrium sound speed')

axes[1, 0].set_xlabel(r'$n/n_0$')
axes[1, 0].set_ylabel(r'$\epsilon/\rho_B - m_N$ [GeV]')
axes[1, 0].set_title('Beta-equilibrium energy per baryon')

axes[1, 1].set_xlabel(r'$n/n_0$')
axes[1, 1].set_ylabel(r'$P$ [MeV fm$^{-3}$]')
axes[1, 1].set_title('Beta-equilibrium pressure')

comparison_png = OUTPUT_ROOT / "clausius_branch_comparison_k0_overlay.png"
comparison_pdf = OUTPUT_ROOT / "clausius_branch_comparison_k0_overlay.pdf"
fig.savefig(comparison_png, dpi=200, bbox_inches="tight")
fig.savefig(comparison_pdf, bbox_inches="tight")
plt.show()

print('saved:', comparison_png)
print('saved:', comparison_pdf)


saved: /Users/dajuarez4/Documents/QuarkMatt/good_repo/QMM/results/generated/clausius_asymmetric_scan_lambda200_smooth/clausius_branch_comparison_k0_overlay.png
saved: /Users/dajuarez4/Documents/QuarkMatt/good_repo/QMM/results/generated/clausius_asymmetric_scan_lambda200_smooth/clausius_branch_comparison_k0_overlay.pdf


/var/folders/vw/wh6xbs2n4y570pctw6_y3t0r0000gp/T/ipykernel_39834/3628767870.py:82: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Fixed-`y` Asymmetric Quarkyonic Profiles

These are the non-beta-equilibrated branches for `y = 0.0, 0.1, ..., 0.5`.


In [13]:
selected_target_k0_fixed_y = target_k0_values[0]
selected_summary_fixed_y = scan_summaries[selected_target_k0_fixed_y]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)

for y_value in proton_fraction_values:
    key = f'asymmetric_y_{y_value:0.3f}_csv'
    csv_path = Path(selected_summary_fixed_y['outputs'][key])
    df = pd.read_csv(csv_path)
    label = f'y={y_value:.1f}'
    axes[0].plot(df['n_over_n0'], df['quark_fraction'], lw=2, label=label)
    axes[1].plot(df['n_over_n0'], df['vs2'], lw=2, label=label)

axes[0].set_xlabel(r'$n/n_0$')
axes[0].set_ylabel(r'$f_Q$')
axes[0].set_title(f'Quark fraction for K0={selected_target_k0_fixed_y:.0f} MeV')
axes[0].legend(loc='upper left', ncol=2)

axes[1].set_xlabel(r'$n/n_0$')
axes[1].set_ylabel(r'$v_s^2$')
axes[1].set_ylim(-0.1, 0.8)
axes[1].axhline(1.0 / 3.0, color='gray', ls=(0, (5, 5)), lw=1.0)
axes[1].set_title(f'Sound speed for K0={selected_target_k0_fixed_y:.0f} MeV')
axes[1].legend(loc='upper right', ncol=2)

plt.show()


/var/folders/vw/wh6xbs2n4y570pctw6_y3t0r0000gp/T/ipykernel_39834/170734410.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Output Check

This final cell lists the generated JSON inputs and output folders for the scan.


In [14]:
print('Generated config files:')
for path in sorted((OUTPUT_ROOT / 'configs').glob('*.json')):
    print(' -', path.relative_to(ROOT))

print('Output folders:')
for path in sorted(OUTPUT_ROOT.glob('K0_*')):
    if path.is_dir():
        print(' -', path.relative_to(ROOT))


Generated config files:
 - results/generated/clausius_asymmetric_scan_lambda200_smooth/configs/clausius_k0_250.json
 - results/generated/clausius_asymmetric_scan_lambda200_smooth/configs/clausius_k0_260.json
 - results/generated/clausius_asymmetric_scan_lambda200_smooth/configs/clausius_k0_270.json
 - results/generated/clausius_asymmetric_scan_lambda200_smooth/configs/clausius_k0_280.json
 - results/generated/clausius_asymmetric_scan_lambda200_smooth/configs/clausius_k0_290.json
 - results/generated/clausius_asymmetric_scan_lambda200_smooth/configs/clausius_k0_300.json
 - results/generated/clausius_asymmetric_scan_lambda200_smooth/configs/clausius_k0_315.json
Output folders:
 - results/generated/clausius_asymmetric_scan_lambda200_smooth/K0_250
 - results/generated/clausius_asymmetric_scan_lambda200_smooth/K0_260
 - results/generated/clausius_asymmetric_scan_lambda200_smooth/K0_270
 - results/generated/clausius_asymmetric_scan_lambda200_smooth/K0_280
 - results/generated/clausius_asymme